In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

문제 1 (단답형 주관식)

- Full Freeze (전체 동결)
- Partial Fine-tuning (부분 해제)
- Full Fine-tuning (전체 해제)

이 세 가지 전략이 각각 어떤 데이터셋 상황(예: 데이터가 많은지/적은지, 원본과 유사한지/다른지)에서 가장 적합한지 간략히 서술하시오.

1번 답을 적어 주세요

1) Full Freeze가 적합한 상황: 데이터 사이즈가 매우 작을 때 (1,000개 이하)



2) Partial Fine-tuning이 적합한 상황: 데이터 수가 1~10,000개 정도로 중간 정도이며, 원본 모델의 도메인과 유사할 때


3) Full Fine-tuning이 적합한 상황: 데이터가 10,000개 이상으로 많을 때

문제 2 (단답형 주관식)

'Full Fine-tuning(전체 해제)' 전략을 사용할 때, 모델의 모든 계층을 동일한 학습률(Learning Rate)로 학습시키지 않고 계층별로 학습률을 다르게 설정하는 "차등 학습률
(Discriminative Learning Rates)" 기법을 적용했습니다.
이 기법을 사용하는 이유를 '백본(Backbone)'과 '분류기(Classifier)'의 관점에서 서술하시오.

백본 층은 이미 사전 학습되어 있기 때문에 큰 변화가 필요하지 않다. 따라서 백본에서는 학습률을 작게 설정한다.

반면 분류기는 사용 용도에 맞게 새로 만든 층이므로, 랜덤으로 초기화되어 큰 변화가 필요하다. 따라서 학습률을 크게 설정한다.

문제 3 (실습 문제 - 코드 빈칸 채우기)

torchvision.models에서 resnet50 모델을 불러온 뒤, "Full Freeze(전체 동결)" 전략을 적용하기 위해 모델의 모든 파라미터를 고정(freeze)하는 코드입니다.

2개의 빈칸 (# TODO: ...)을 채워 파라미터 고정 로직을 완성하시오.

In [ ]:
import torchvision.models as models

# --- 사전 학습된 ResNet-50 모델 로드 (수정 불필요) ---
model = models.resnet50(pretrained=True)
# ------------------------------------------------

print(f"변경 전 (예시: layer1): {model.layer1[0].conv1.weight.requires_grad}")
print(f"변경 전 (예시: fc): {model.fc.weight.requires_grad}")

# TODO: 1. model의 모든 파라미터를 순회(loop)
for param in model.parameters():

    # TODO: 2. 각 파라미터(param)의 requires_grad를 False로 설정하여 고정
    param.requires_grad = False


# --- 결과 확인 (수정 불필요) ---
print(f"\n변경 후 (예시: layer1): {model.layer1[0].conv1.weight.requires_grad}")
print(f"변경 후 (예시: fc): {model.fc.weight.requires_grad}")
# (참고: 이후 fc는 새 레이어로 교체되므로 True가 됩니다)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 137MB/s]


변경 전 (예시: layer1): True
변경 전 (예시: fc): True

변경 후 (예시: layer1): False
변경 후 (예시: fc): False


문제 4 (실습 문제 - 코드 빈칸 채우기)

"Full Freeze" 전략을 위해 사전 학습 모델의 마지막 fc 계층을 새로운 분류기(Classifier)로 교체하는 코드입니다.

(ResNet-50 기준)

3개의 빈칸 (# TODO: ...)을 채워 마지막 fc 레이어를 교체하시오.

In [ ]:
import torch.nn as nn
import torchvision.models as models

# --- 사전 학습된 ResNet-50 모델 로드 (수정 불필요) ---
model = models.resnet50(pretrained=True)
print(f"원본 FC 레이어:\n{model.fc}\n")
# ------------------------------------------------

# TODO: 1. ResNet-50의 'fc' 계층의 입력 특성 수(in_features) 가져오기
n_features = model.fc.in_features

# TODO: 2. 새로운 출력 클래스 수
num_classes = 5 # (강의 예제와 동일하게 5개로 가정)

# TODO: 3. model.fc를 n_features 입력, num_classes 출력을 갖는 새로운 nn.Linear 계층으로 교체
model.fc = nn.Linear(n_features, num_classes)


# --- 결과 확인 (수정 불필요) ---
print(f"변경된 FC 레이어:\n{model.fc}")
# (참고: 새로 정의된 레이어는 requires_grad가 True입니다)
print(f"변경된 FC 레이어의 학습 여부: {model.fc.weight.requires_grad}")

원본 FC 레이어:
Linear(in_features=2048, out_features=1000, bias=True)

변경된 FC 레이어:
Linear(in_features=2048, out_features=5, bias=True)
변경된 FC 레이어의 학습 여부: True


문제 5 (실습 문제 - 코드 빈칸 채우기)

"Full Freeze(전체 동결)" 전략을 위한 옵티마이저를 정의하는 코드입니다.

이 전략은 새로 교체된 model.fc 계층의 파라미터만 학습해야 합니다.

1개의 빈칸 (# TODO: ...)을 채워 옵티마이저가 model.fc.parameters()만 학습하도록 설정하시오.

In [ ]:
import torch.optim as optim
import torch.nn as nn
import torchvision.models as models

# --- 모델 준비 (수정 불필요) ---
model = models.resnet50(pretrained=True)
# (모든 파라미터 고정 가정)
for param in model.parameters():
    param.requires_grad = False
# (분류기 교체)
model.fc = nn.Linear(model.fc.in_features, 5)
base_lr = 0.01
# ---------------------------------

optimizer = optim.SGD(

    # TODO: 1. 학습할 파라미터로 'model.fc.parameters()'만 지정
    model.fc.parameters(),

    lr=base_lr,
    momentum=0.9
)

# --- 결과 확인 (수정 불필요) ---
print("옵티마이저가 학습할 파라미터 그룹:")
for param_group in optimizer.param_groups:
    print(f"Learning Rate: {param_group['lr']}")
    print(f"파라미터 수: {len(param_group['params'])}") # 2개 (weight, bias)

옵티마이저가 학습할 파라미터 그룹:
Learning Rate: 0.01
파라미터 수: 2


문제 6 (실습 문제 - 코드 작성)

"Full Fine-tuning(전체 해제)" 전략을 위해 차등 학습률(Discriminative Learning Rates)을 적용하는 옵티마이저를 정의하는 코드입니다.

[요구사항]

optim.SGD의 파라미터 리스트에 3개의 딕셔너리를 전달하여 다음과 같이 학습률을 차등 적용하시오.

- model.fc 파라미터: 학습률 base_lr (1.0배)
- model.layer4 파라미터: 학습률 base_lr * 0.1 (0.1배)
- 그 외 모든 파라미터: 학습률 base_lr * 0.01 (0.01배)

(힌트: model.parameters()는 제너레이터(generator)이므로 한 번 순회하면 비워집니다. other_params 정의 시 model.fc와 model.layer4의 파라미터를 제외해야 합니다.)

In [ ]:
import torch.optim as optim
import torch.nn as nn
import torchvision.models as models

# --- 모델 준비 (수정 불필요) ---
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 5)
base_lr = 0.01
# ---------------------------------

# TODO: 1. model.fc와 model.layer4의 파라미터 ID 저장
fc_params = list(model.fc.parameters())
layer4_params = list(model.layer4.parameters())


# TODO: 2. fc와 layer4를 제외한 'other_params' 리스트 생성
other_params = [param for name, param in model.named_parameters()
                if ('fc' not in name and 'layer4' not in name)
                ]
                # (param not in fc_params) and (param not in layer4_params)]
                # 이렇게 param을 직접 비교하면 param과 fc_params shape 안맞아서 에러 뜸

# TODO: 3. 차등 학습률을 적용하기 위한 'params_to_optimize' 리스트 정의
params_to_optimize = [
    # (그룹 1: model.fc 파라미터)
    {'params': fc_params, 'lr': base_lr},

    # (그룹 2: model.layer4 파라미터)
    {'params': layer4_params, 'lr': 0.1 * base_lr},

    # (그룹 3: other_params)
    {'params': other_params, 'lr': 0.01 * base_lr}
]

# TODO: 4. params_to_optimize를 optim.SGD에 전달
optimizer = optim.SGD(
    params_to_optimize,
    momentum=0.9
)


# --- 결과 확인 (수정 불필요) ---
print("옵티마이저가 학습할 파라미터 그룹:")
for i, param_group in enumerate(optimizer.param_groups):
    print(f"\n[그룹 {i+1}]")
    print(f"Learning Rate: {param_group['lr']:.4f}")
    print(f"파라미터 수: {len(param_group['params'])}")

옵티마이저가 학습할 파라미터 그룹:

[그룹 1]
Learning Rate: 0.0100
파라미터 수: 2

[그룹 2]
Learning Rate: 0.0010
파라미터 수: 30

[그룹 3]
Learning Rate: 0.0001
파라미터 수: 129


[자율학습]

### 14_01 전이학습

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import numpy as np
import random

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

if torch.cuda.is_available():
  torch.cuda.manual_seed(0)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    ## transforms.RandomResizedCrop: 무작위로 자르고, 지정한 크기로 리사이즈
    # size: 보간법으로 크기를 줄이거나 키움
    # scale: 원본의 60~100% 면적을 랜덤하게 선택

    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std = (0.229, 0.224, 0.225))
    ## transforms.Normalize
    # 데이터를 평균 0, 표준편차 1로 만들기 위해 진행
    # mean, std는 RGB 각 채널의 평균, 표준편차
    # 왜 저런 수치값?
    # 데이터셋의 평균, 표준편차값을 미리 계산해둔 값임
])

transform_test = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std = (0.229, 0.224, 0.225))
])

In [ ]:
dataset_train_full = datasets.CIFAR10(
    root = '/tmp/cifar.tl',
    train = True,
    download = True,
    transform = transform_train
)

dataset_test = datasets.CIFAR10(
    root = '/tmp/cifar.tl',
    train = False,
    download = True,
    transform = transform_test
)

print(f'전체 학습 데이터: {len(dataset_train_full):,}개')
print(f'전체 테스트 데이터: {len(dataset_test):,}개')
print(f'클래스 수: 10개 (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck)')

100%|██████████| 170M/170M [00:03<00:00, 53.4MB/s]


전체 학습 데이터: 50,000개
전체 테스트 데이터: 10,000개
클래스 수: 10개 (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck)


소규모 서브셋 생성 (클래스당 데이터 조금씩 빼오기)

In [ ]:
selected_indices = []   # 선택된 데이터의 인덱스 넣기

class_counts = {i: 0 for i in range(10)}  # 클래스별로 데이터 개수 카운트

for idx, (image, label) in enumerate(dataset_train_full):
  if class_counts[label] < 300:
    selected_indices.append(idx)
    class_counts[label] += 1

  if len(selected_indices) >= 3000:
    break

## 가장 핵심 부분 Subset(원본 dataset, 가져올 인덱스 리스트)
dataset_train_small = Subset(dataset_train_full, selected_indices)

print(len(dataset_train_small))

for class_id, count in class_counts.items():
  class_name = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                'dog', 'frog', 'horse', 'ship', 'truck'][class_id]
  print(f'{class_id} ({class_name}): {count}개')

3000
0 (airplane): 300개
1 (automobile): 300개
2 (bird): 300개
3 (cat): 300개
4 (deer): 300개
5 (dog): 300개
6 (frog): 300개
7 (horse): 300개
8 (ship): 300개
9 (truck): 300개


In [ ]:
train_loader = DataLoader(
    dataset_train_small,
    batch_size = 64,
    shuffle = True,

    ## GPU 병렬 처리 설정
    num_workers = 2,    # 데이터 로딩할 때 몇 개의 CPU 프로세스로 병렬 처리할 지
    pin_memory = True   # CPU가 고정된 버퍼를 사용하여 GPU로 빠르게 데이터 복사 가능
)

test_loader = DataLoader(
    dataset_test,
    batch_size = 128,
    shuffle = False,
    num_workers = 2,
    pin_memory = True
)

# 데이터 로더 정보 출력
print(f'\n학습 데이터:')
print(f'  - 총 샘플 수: {len(dataset_train_small):,}개')
print(f'  - 배치 크기: 64')
print(f'  - 배치 수: {len(train_loader)}개')
print(f'\n테스트 데이터:')
print(f'  - 총 샘플 수: {len(dataset_test):,}개')
print(f'  - 배치 크기: 128')
print(f'  - 배치 수: {len(test_loader)}개')


학습 데이터:
  - 총 샘플 수: 3,000개
  - 배치 크기: 64
  - 배치 수: 47개

테스트 데이터:
  - 총 샘플 수: 10,000개
  - 배치 크기: 128
  - 배치 수: 79개


In [ ]:
def build_model(strategy='freeze'):
  model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

  in_features = model.fc.in_features

  model.fc = nn.Linear(in_features, 10)

  if strategy == 'freeze':
    print('[freeze 전략] 백본 전체를 동결 >> 분류기만 학습')

    for param in model.layer1.parameters():
      param.requires_grad = False
    for param in model.layer2.parameters():
      param.requires_grad = False
    for param in model.layer3.parameters():
      param.requires_grad = False
    for param in model.layer4.parameters():
      param.requires_grad = False

  elif strategy == 'partial':
    print('[partial 전략] 마지막 블록(layer4)과 분류기만 학습')

    for param in model.parameters():
      param.requires_grad = False

    for param in model.layer4.parameters():
      param.requires_grad = True
    for param in model.fc.parameters():
      param.requires_grad = True

  elif strategy == 'full':
    print('[full 전략]: 모든 층을 학습')

    for param in model.parameters():
      param.requires_grad = True

  else:
    raise ValueError('지원되지 않는 전략입니다.')

  model.to(device)

  ## numel 함수: 텐서의 길이 (리스트의 len같은 함수)
  trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
  total_params = sum(p.numel() for p in model.parameters())
  print(f'학습 가능한 파라미터: {trainable_params:,} / {total_params:,}')

  return model



In [ ]:
def train_and_evaluate(strategy):
  model = build_model(strategy)

  head_params = list(model.fc.parameters())

  backbone_params = [
      param for name, param in model.named_parameters()
      if 'fc' not in name and param.requires_grad
  ]


  param_groups = []

  ## if문 사용하는 이유?
  #
  if backbone_params:
    param_groups.append({
        'params': backbone_params,
        'lr': 1e-4
    })

  if head_params:
    param_groups.append({
        'params': head_params,
        'lr': 1e-3
    })

  ## param_groups
  # layer마다 학습률을 다르게 하고 싶으니까,
  # 각 layer의 param과 학습률을 묶어서 optimizer에 보내자
  optimizer = optim.AdamW(param_groups, weight_decay = 1e-4)

  criterion = nn.CrossEntropyLoss()

  num_epoch = 10

  for epoch in range(num_epoch):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, labels) in enumerate(train_loader):
      inputs = inputs.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()

      outputs = model(inputs)

      loss = criterion(outputs, labels)

      loss.backward()

      optimizer.step()

      running_loss += loss.item() * inputs.size(0)
      ## 여기서 inputs.size(0)을 곱해주는 이유?
      # 현재 '배치 단위'로 데이터를 가져와서, '배치 단위' 손실, 정확도를 계산하고 있음
      # 따라서 각 배치마다 loss * batch_size로 total 계산, 이를 전체 표본 수로 나눠야함

      # 하나의 epoch 안에서 배치 사이즈가 달라지는 경우를 대비하기 위함
      # 전체 데이터 개수 N이 배치 사이즈로 나누어떨어지지 않는 경우,
      # 마지막 배치가 남음

      _, predicted = outputs.max(1)

      total += labels.size(0)     # inputs.size(0) 써도 됨 (둘 다 batch size를 나타내기 위함)
      correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    print(f'Epoch [({epoch+1}/{num_epoch})]'
          f'Loss: {epoch_loss:.4f},'
          f'Train_Acc: {epoch_acc:.2f}%'
    )

    model.eval()

    with torch.no_grad():
      for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        predicted = outputs.argmax(dim=1)

        ## predicted 구하는 다양한 문법
        # predicted = outputs.argmax(dim=1)
        # predicted = torch.max(outputs, 1)[1]
        # predicted = outputs.max(1)[1]

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

  test_accuracy = correct / total

  print(f'테스트 정확도: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

  return test_accuracy


In [ ]:
acc_freeze = train_and_evaluate('freeze')

acc_partial = train_and_evaluate('partial')

acc_full = train_and_evaluate('full')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


[freeze 전략] 백본 전체를 동결 >> 분류기만 학습
학습 가능한 파라미터: 14,666 / 11,181,642


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch [(1/10)]Loss: 1.8831,Train_Acc: 37.10%


KeyboardInterrupt: 

In [ ]:
results = {
    'Freeze (백본 전체 동결)': acc_freeze,
    'Partial (layer4, fc 해제)': acc_partial,
    'Full (모든 층 학습)': acc_full
}

print(f'\n{'전략':<30} {'테스트 정확도':>15}')
print('='*70)

for strategy_name, accuracy in results.items():
  print(f'{strategy_name:<30} {accuracy*100:>14.2f}%')

  ## f-string 출력 폭, 정렬 방향 지정
  # :<30 왼쪽 정렬, 30칸 차지
  # :>15 오른쪽 정렬, 15칸 차지
  # :>14.2f 오른쪽 정렬, 총 14칸, 소수점 둘째 자리까지 표시